In [13]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

# DataSets

In [14]:
mnist_dataset, mnist_info = tfds.load(name="mnist", with_info=True, as_supervised=True)

In [15]:
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

num_of_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_of_validation_samples = tf.cast(num_of_validation_samples, tf.int64)

num_of_test_samples = mnist_info.splits['test'].num_examples
num_of_test_samples = tf.cast(num_of_test_samples, tf.int64)

# scale data to numerically stable
def scale(image, label):
    image = tf.cast(image, tf.float32)
    image = image/ 255.
    return image, label

scale_train_validation_data = mnist_train.map(scale)
scale_test_data = mnist_test.map(scale)

# shuffle the data
BUFFER_SIZE = 10000
shuffle_scale_train_validation_data =scale_train_validation_data.shuffle(BUFFER_SIZE)
validation_data = shuffle_scale_train_validation_data.take(num_of_validation_samples)
training_data = shuffle_scale_train_validation_data.skip(num_of_validation_samples)

BATCH_SIZE = 100
training_data = training_data.batch(BATCH_SIZE)
validation_data = validation_data.batch(num_of_validation_samples)
test_data = mnist_test.batch(num_of_test_samples)

# MNIST data is iterable and 2-tuple format as we set as_supervised=True
# we must extract and covert validation_inputs, validation_targets
validation_inputs, validation_targets = next(iter(validation_data))



# Model
## Outline

In [16]:
input_size = 784 # 28*28
output_size = 10
hidden_layer_size = 50

# tf.keras.layers.Dense() : takes the inputs provided to the model,
# calculate dot product of the inputs and weights add the bias
# This is where we can apply activation function

mnist_model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(hidden_layer_size, activation=tf.nn.relu), # hidden layer 1
    tf.keras.layers.Dense(hidden_layer_size, activation=tf.nn.relu), # hidden layer 1
    tf.keras.layers.Dense(output_size, activation=tf.nn.softmax), # Output we use softmax as activation function of the output layer must transform values into probabilities

])

C:\shiva\coding\python_projects\dear_comrade_data_science_zero_to_hero\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# Choosing optimizer and loss function
## One of the best choise we got is Adam(adaptive movement estaimation)

In [17]:
mnist_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    metrics=['accuracy']
)